# Part B — Data Cleaning

This part focuses on cleaning and validating the finalized HI-Small datasets before storing them in PostgreSQL and using them for feature engineering, graph analysis, and machine learning.

## B.1 Load Data

Load the finalized datasets:

- `HI-Small_accounts.csv`
- `HI-Small_Trans_Custom.csv`

The transaction dataset is the combined dataset created from `HI-Small_Trans_half.csv` and `HI-Small_Trans_balanced.csv`, with duplicate records removed.

## B.2 Create Backup

Create a backup copy of the original loaded datasets before performing any cleaning operations.

This ensures that the original data remains unchanged and can be restored or compared if required.

## B.3 Clean Accounts Dataset

### B.3.1 Duplicate Records

Check the accounts dataset for duplicate records and remove duplicates where necessary.

### B.3.2 Missing Values

Check for missing or null values in the accounts dataset and handle them appropriately.

### B.3.3 Account ID Validation

Validate the `Account Number` column to ensure that account identifiers are present, unique where required, and stored in a consistent format.

### B.3.4 Data Type Validation

Check and correct the data types of the accounts dataset columns to ensure they are suitable for PostgreSQL storage and further analysis.

## B.4 Clean Transaction Dataset

### B.4.1 Duplicate Records

Check the transaction dataset for duplicate records and remove duplicates where necessary.

### B.4.2 Missing Values

Identify missing or null values in transaction records and handle them according to the requirements of the project.

### B.4.3 Timestamp

Validate the `Timestamp` column and convert it into a consistent datetime format for time-based analysis and feature engineering.

### B.4.4 Amount Validation

Validate transaction amounts to identify invalid, negative, zero, or inconsistent values in:

- `Amount Received`
- `Amount Paid`

### B.4.5 Sender/Receiver Validation

Validate the sender and receiver account information.

The transaction dataset uses:

- `Account` → `Sender_Account`
- `Account.1` → `Receiver_Account`

Verify that the referenced sender and receiver accounts exist in the accounts dataset.

### B.4.6 Target Validation

Validate the `Is Laundering` column, which is the target variable for the fraud/AML detection models.

Check that the target contains valid values and understand the distribution of the target classes before model training.

## B.5 Validate Relationship

Validate the relationship between the accounts and transaction datasets.

Ensure that:

- Every sender account exists in the accounts dataset.
- Every receiver account exists in the accounts dataset.
- Account identifiers use a consistent format.
- No invalid account references remain after cleaning.

## B.6 Final Cleaning Summary

Summarize the cleaning process and compare the datasets before and after cleaning.

The summary should include:

- Number of records before cleaning
- Number of duplicate records removed
- Number of missing values handled
- Number of invalid records removed
- Number of records after cleaning
- Number of accounts after cleaning
- Number of transactions after cleaning

## B.7 Save Clean Datasets

Save the cleaned datasets as separate CSV files.

Final output files:

- `cleaned_HI-Small_accounts.csv`
- `cleaned_HI-Small_Trans_Custom.csv`

These cleaned datasets will be used in the next stages of the project, including:

1. PostgreSQL Database Storage
2. Feature Engineering
3. Graph Analysis
4. Machine Learning
5. Ensemble Model
6. Fraud Detection Dashboard

In [2]:
import pandas as pd
import numpy as np
import os

In [5]:
accounts_path = r"D:\Projects\major project\FraudLens-AI\Data\Raw_Data\HI-Small_accounts.csv"
transactions_path = r"D:\Projects\major project\FraudLens-AI\Data\Raw_Data\HI-Small_Trans_Custom.csv"

accounts = pd.read_csv(accounts_path)
transactions = pd.read_csv(transactions_path)

In [6]:
print("Accounts:", accounts.shape)
print("Transactions:", transactions.shape)

Accounts: (518581, 5)
Transactions: (2505191, 11)


In [7]:
accounts_raw = accounts.copy()
transactions_raw = transactions.copy()

In [8]:
accounts.duplicated().sum()

np.int64(0)

In [9]:
accounts.isnull().sum()

Bank Name         0
Bank ID           0
Account Number    0
Entity ID         0
Entity Name       0
dtype: int64

In [10]:
accounts_missing = pd.DataFrame({
    "Attribute": accounts.columns,
    "Missing_Count": accounts.isna().sum().values,
    "Missing_Percentage": (
        accounts.isna().mean().values * 100
    ).round(2)
})

accounts_missing

,Attribute,Missing_Count,Missing_Percentage
0,Bank Name,0,0.0
1,Bank ID,0,0.0
2,Account Number,0,0.0
3,Entity ID,0,0.0
4,Entity Name,0,0.0


In [11]:
accounts.columns.tolist()

['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name']

In [13]:
print("Total accounts:", len(accounts))
print("Unique accounts:", accounts["Account Number"].nunique())

Total accounts: 518581
Unique accounts: 518573


In [15]:
print(
    "Missing Account IDs:",
    accounts["Account Number"].isna().sum()
)


Missing Account IDs: 0


In [17]:
accounts["Account Number"].duplicated().sum()

np.int64(8)

In [18]:
transactions.duplicated().sum()

np.int64(0)

In [20]:
transactions_missing = pd.DataFrame({
    "Attribute": transactions.columns,
    "Missing_Count": transactions.isna().sum().values,
    "Missing_Percentage": (
        transactions.isna().mean().values * 100
    ).round(2)
})

transactions_missing

,Attribute,Missing_Count,Missing_Percentage
0,Timestamp,0,0.0
1,From Bank,0,0.0
2,Account,0,0.0
3,To Bank,0,0.0
4,Account.1,0,0.0
5,Amount Received,0,0.0
6,Receiving Currency,0,0.0
7,Amount Paid,0,0.0
8,Payment Currency,0,0.0
9,Payment Format,0,0.0


In [21]:
transactions["Timestamp"].head()

0    2022/09/01 00:20
1    2022/09/01 00:20
2    2022/09/01 00:00
3    2022/09/01 00:02
4    2022/09/01 00:06
Name: Timestamp, dtype: str

In [22]:
transactions["Timestamp"] = pd.to_datetime(
    transactions["Timestamp"],
    errors="coerce"
)

In [23]:
transactions["Timestamp"].isna().sum()

np.int64(0)

In [24]:
transactions.columns.tolist()

['Timestamp',
 'From Bank',
 'Account',
 'To Bank',
 'Account.1',
 'Amount Received',
 'Receiving Currency',
 'Amount Paid',
 'Payment Currency',
 'Payment Format',
 'Is Laundering']

In [25]:
transactions[
    ["Amount Paid", "Amount Received"]
].describe()

,Amount Paid,Amount Received
count,2.505191e+06,2.505191e+06
mean,5.267170e+06,7.050758e+06
std,7.082822e+08,9.748583e+08
min,1.000000e-06,1.000000e-06
25%,1.668300e+02,1.657800e+02
50%,1.600120e+03,1.595820e+03
75%,1.577865e+04,1.583693e+04
max,6.260355e+11,6.260355e+11


In [27]:
print(
    "Negative Amount_Paid:",
    (transactions["Amount Paid"] < 0).sum()
)

print(
    "Negative Amount_Received:",
    (transactions["Amount Received"] < 0).sum()
)

Negative Amount_Paid: 0
Negative Amount_Received: 0


In [29]:
print(
    "Zero Amount_Paid:",
    (transactions["Amount Paid"] == 0).sum()
)

Zero Amount_Paid: 0


In [30]:
transactions.columns.tolist()

['Timestamp',
 'From Bank',
 'Account',
 'To Bank',
 'Account.1',
 'Amount Received',
 'Receiving Currency',
 'Amount Paid',
 'Payment Currency',
 'Payment Format',
 'Is Laundering']

In [31]:
print("Missing sender accounts:",
      transactions["Account"].isna().sum())

print("Missing receiver accounts:",
      transactions["Account.1"].isna().sum())

Missing sender accounts: 0
Missing receiver accounts: 0


In [32]:
print("Unique senders:",
      transactions["Account"].nunique())

print("Unique receivers:",
      transactions["Account.1"].nunique())

Unique senders: 491611
Unique receivers: 419428


In [36]:
transactions = transactions.rename(columns={
    "Account": "Sender_Account",
    "Account.1": "Receiver_Account"
})

In [40]:
print("Accounts columns:")
print(accounts.columns.tolist())

print("\nTransaction columns:")
print(transactions.columns.tolist())

Accounts columns:
['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name']

Transaction columns:
['Timestamp', 'From Bank', 'Sender_Account', 'To Bank', 'Receiver_Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


In [44]:
account_ids = set(
    accounts["Account Number"].dropna().unique()
)

sender_ids = set(
    transactions["Sender_Account"].dropna().unique()
)

receiver_ids = set(
    transactions["Receiver_Account"].dropna().unique()
)

In [45]:
sender_not_found = sender_ids - account_ids
receiver_not_found = receiver_ids - account_ids

print(
    "Sender accounts not found:",
    len(sender_not_found)
)

print(
    "Receiver accounts not found:",
    len(receiver_not_found)
)

Sender accounts not found: 0
Receiver accounts not found: 0


In [46]:
transactions.columns.tolist()

['Timestamp',
 'From Bank',
 'Sender_Account',
 'To Bank',
 'Receiver_Account',
 'Amount Received',
 'Receiving Currency',
 'Amount Paid',
 'Payment Currency',
 'Payment Format',
 'Is Laundering']

In [47]:
transactions["Is Laundering"].value_counts(
    dropna=False
)

Is Laundering
0    2501229
1       3962
Name: count, dtype: int64

In [49]:
transactions["Is Laundering"].value_counts(
    normalize=True,
    dropna=False
) * 100

Is Laundering
0    99.841848
1     0.158152
Name: proportion, dtype: float64

In [51]:
transactions["Payment Format"].value_counts(
    dropna=False
)

Payment Format
Cheque          809519
Credit Card     577083
Reinvestment    481056
ACH             275224
Cash            215016
Wire             76157
Bitcoin          71136
Name: count, dtype: int64

In [52]:
transactions["Payment Currency"].value_counts(
    dropna=False
)

Payment Currency
US Dollar            937110
Euro                 577074
Swiss Franc          116065
Yuan                 105093
Shekel                94346
Rupee                 93765
UK Pound              88635
Yen                   76895
Ruble                 76436
Bitcoin               71124
Canadian Dollar       68645
Australian Dollar     67367
Mexican Peso          54659
Saudi Riyal           43311
Brazil Real           34666
Name: count, dtype: int64

In [53]:
transactions["Receiving Currency"].value_counts(
    dropna=False
)

Receiving Currency
US Dollar            930325
Euro                 578754
Swiss Franc          117387
Yuan                 101953
Shekel                95558
Rupee                 94561
UK Pound              88829
Yen                   77377
Ruble                 77359
Bitcoin               72017
Canadian Dollar       69170
Australian Dollar     68124
Mexican Peso          55016
Saudi Riyal           43738
Brazil Real           35023
Name: count, dtype: int64

In [54]:
print("===== FINAL DATASET CHECK =====")

print("Accounts shape:", accounts.shape)
print("Transactions shape:", transactions.shape)

print("\nAccounts duplicates:",
      accounts.duplicated().sum())

print("Transaction duplicates:",
      transactions.duplicated().sum())

===== FINAL DATASET CHECK =====
Accounts shape: (518581, 5)
Transactions shape: (2505191, 11)

Accounts duplicates: 0
Transaction duplicates: 0


In [55]:
print("\nAccounts missing values:")
print(accounts.isna().sum())

print("\nTransactions missing values:")
print(transactions.isna().sum())


Accounts missing values:
Bank Name         0
Bank ID           0
Account Number    0
Entity ID         0
Entity Name       0
dtype: int64

Transactions missing values:
Timestamp             0
From Bank             0
Sender_Account        0
To Bank               0
Receiver_Account      0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64


In [56]:
cleaning_summary = pd.DataFrame({
    "Dataset": [
        "Accounts",
        "Transactions"
    ],
    "Original_Rows": [
        len(accounts_raw),
        len(transactions_raw)
    ],
    "Final_Rows": [
        len(accounts),
        len(transactions)
    ]
})

cleaning_summary["Rows_Removed"] = (
    cleaning_summary["Original_Rows"]
    - cleaning_summary["Final_Rows"]
)

cleaning_summary

,Dataset,Original_Rows,Final_Rows,Rows_Removed
0,Accounts,518581,518581,0
1,Transactions,2505191,2505191,0


In [57]:
os.makedirs("../data/processed", exist_ok=True)

In [58]:
accounts.to_csv(
    "../data/processed/accounts_clean.csv",
    index=False
)

transactions.to_csv(
    "../data/processed/transactions_clean.csv",
    index=False
)